# 09 · Agente: um LLM que usa a rede neural como ferramenta

Os notebooks anteriores treinaram uma rede que prevê risco a partir de 11 dados
estruturados. Na prática, a equipe de saúde descreve o paciente em **texto livre**:
"homem, 58 anos, pressão 15 por 9, fumante…". Este notebook liga as duas coisas com um
**agente ReAct** (LangChain + OpenAI), como nos slides de LangChain:

1. **Raciocinar** — o LLM lê o texto e decide o que fazer (perguntar o que falta? chamar
   uma ferramenta?);
2. **Agir** — chama uma ferramenta Python;
3. **Observar** — lê o resultado da ferramenta;
4. **Responder** — explica em linguagem simples.

**Divisão de trabalho:** o LLM entende linguagem, mas **não calcula risco**. Todo número
vem da rede neural treinada no notebook 05. Isso evita que o LLM invente probabilidades e
mantém a previsão auditável.

| Ferramenta | O que faz |
|:--|:--|
| `converter_pressao` | converte a pressão escrita à brasileira ("15 por 9,5") para mmHg, em código e não pelo LLM |
| `avaliar_paciente` | chama a rede: probabilidade, decisão de triagem, fatores que mais pesaram, avisos de extrapolação |
| `desempenho_do_modelo` | métricas no teste (AUC, sensibilidade, especificidade) |
| `consultar_achados` | busca nos achados documentados (paradoxo do fumo, limiar, limitações…) |

O código está em `src/agente/`. Sem `OPENAI_API_KEY`, o notebook roda as ferramentas e um
LLM roteirizado; com a chave (arquivo `.env`, veja `.env.exemplo`), roda conversas reais.

In [1]:
import sys
from pathlib import Path

# Permite importar `src` mesmo fora do ambiente do uv (ex.: Colab com o repositório clonado).
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import json
import os

import pandas as pd

from src.agente import ferramentas
from src.agente.agente import INSTRUCOES, conversar, criar_agente

TEM_CHAVE = bool(os.getenv("OPENAI_API_KEY"))
print("OPENAI_API_KEY encontrada: conversas reais serão executadas." if TEM_CHAVE
      else "Sem OPENAI_API_KEY: as seções com LLM real serão puladas.")

OPENAI_API_KEY encontrada: conversas reais serão executadas.


## 1. As ferramentas, sem LLM

Cada ferramenta é uma função Python comum. O decorador `@tool` do LangChain transforma a
assinatura e a docstring em um **esquema** que o LLM lê para saber quando e como chamá-la.
É por isso que as docstrings são escritas para o LLM ("use SEMPRE esta ferramenta para
qualquer número de risco").

In [2]:
esquema = ferramentas.avaliar_paciente.args_schema.model_json_schema()
pd.DataFrame(esquema["properties"]).T[["type", "enum"]].fillna("")

,type,enum
idade,number,
sexo,string,"[feminino, masculino]"
altura_cm,number,
peso_kg,number,
pressao_sistolica,integer,
pressao_diastolica,integer,
colesterol,integer,"[1, 2, 3]"
glicose,integer,"[1, 2, 3]"
fumante,boolean,
consome_alcool,boolean,


In [3]:
paciente = dict(idade=58, sexo="masculino", altura_cm=172, peso_kg=94,
                pressao_sistolica=150, pressao_diastolica=95, colesterol=2, glicose=1,
                fumante=True, consome_alcool=False, fisicamente_ativo=False)
print(json.dumps(ferramentas.avaliar(**paciente), indent=2, ensure_ascii=False))

{
  "probabilidade": 0.849,
  "limiar_de_triagem": 0.382,
  "decisao": "encaminhar para investigação",
  "imc": 31.8,
  "fatores_principais": [
    {
      "variavel": "Pressão sistólica",
      "valor_do_paciente": "150",
      "valor_tipico_da_base": "120",
      "efeito_pontos_percentuais": 22.5,
      "sentido": "aumenta o risco"
    },
    {
      "variavel": "Colesterol",
      "valor_do_paciente": "acima do normal",
      "valor_tipico_da_base": "normal",
      "efeito_pontos_percentuais": 3.0,
      "sentido": "aumenta o risco"
    },
    {
      "variavel": "Peso (kg)",
      "valor_do_paciente": "94",
      "valor_tipico_da_base": "72",
      "efeito_pontos_percentuais": 2.1,
      "sentido": "aumenta o risco"
    },
    {
      "variavel": "Fisicamente ativo",
      "valor_do_paciente": "não",
      "valor_tipico_da_base": "sim",
      "efeito_pontos_percentuais": -1.8,
      "sentido": "reduz o risco"
    },
    {
      "variavel": "Pressão diastólica",
      "valor_do_paci

As ferramentas também protegem o LLM de erros comuns: recusam pressão diastólica maior
que a sistólica e avisam quando um valor está fora da faixa vista no treino (a rede foi
treinada com adultos de 30 a 65 anos).

In [4]:
print(ferramentas.avaliar(**{**paciente, "pressao_diastolica": 160}))
print(ferramentas.avaliar(**{**paciente, "idade": 78})["avisos"])
print(ferramentas.buscar_achados("fumar protege?")[0]["achado"])

{'erro': 'A diastólica precisa ser menor que a sistólica. Confirme os valores.'}
['idade = 78 está fora da faixa vista no treino (30–65); a previsão é uma extrapolação e merece menos confiança.']
Na base, fumo e álcool autodeclarados aparecem associados a risco MENOR. É um artefato conhecido desta base (hábitos autodeclarados e confundidos com sexo e idade), não um efeito protetor. Fumar e beber continuam sendo fatores de risco cardiovascular estabelecidos; o modelo apenas reflete um viés dos dados.


## 2. O ciclo ReAct com um LLM roteirizado

Para ver o mecanismo sem custo de API, trocamos o LLM por um modelo falso que devolve
respostas prontas: primeiro uma **chamada de ferramenta**, depois a resposta final. O
agente de verdade (LangGraph por baixo do `create_agent`) executa a ferramenta entre as
duas etapas e devolve o resultado ao LLM.

In [5]:
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage


class LLMRoteirizado(FakeMessagesListChatModel):
    def bind_tools(self, tools, **kwargs):
        return self


roteiro = LLMRoteirizado(responses=[
    AIMessage(content="", tool_calls=[{"name": "avaliar_paciente", "args": paciente, "id": "1"}]),
    AIMessage(content="(resposta roteirizada) Risco alto: encaminhar para investigação."),
])
resposta, chamadas = conversar(criar_agente(roteiro), [{"role": "user", "content": "Homem, 58 anos..."}])
for c in chamadas:
    print(f"→ agiu: {c['ferramenta']}")
    print(f"← observou: {c['resultado'][:160]}...")
print(f"\nrespondeu: {resposta}")

→ agiu: avaliar_paciente
← observou: {"probabilidade": 0.849, "limiar_de_triagem": 0.382, "decisao": "encaminhar para investigação", "imc": 31.8, "fatores_principais": [{"variavel": "Pressão sistól...

respondeu: (resposta roteirizada) Risco alto: encaminhar para investigação.


## 3. Instruções do agente (prompt de sistema)

O comportamento do agente é definido em linguagem natural: o que extrair, quando
perguntar, o que nunca fazer (estimar risco sem a ferramenta, prescrever tratamento) e
como tratar o paradoxo do fumo.

In [6]:
print(INSTRUCOES)

Você é um assistente de triagem cardiovascular para uma equipe de saúde, em português do Brasil.

Como trabalhar:
1. Extraia do texto os 11 dados do paciente: idade, sexo, altura (cm), peso (kg), pressão
   sistólica e diastólica (mmHg), colesterol e glicose (1 = normal, 2 = acima do normal,
   3 = muito acima do normal), fumante, consome álcool e fisicamente ativo.
   - Pressão: passe o trecho exatamente como foi escrito ("15 por 9,5", "12x8") para a
     ferramenta `converter_pressao` e use os valores que ela devolver. Não converta de cabeça.
   - Altura "1,72 m" significa 172 cm.
   - "Normal" = 1, "acima do normal" / "alto" = 2, "muito acima do normal" / "muito alto" = 3.
   - "Sedentário" = não é fisicamente ativo; "não fuma", "não bebe" = não.
   - Essas conversões são diretas: aplique-as sem pedir confirmação.
   - Se algum dado realmente não foi informado, pergunte por ele. Não invente nem assuma
     valores ausentes.
2. Todo número de risco vem da ferramenta `avaliar_paciente

## 4. Conversas reais (requer `OPENAI_API_KEY`)

Três situações: um paciente completo em linguagem coloquial, um paciente com dados
faltando (o agente deve perguntar, não inventar) e uma pergunta sobre o modelo.

In [7]:
casos = {
    "paciente completo": "Homem de 58 anos, 1,72 m e 94 kg, pressão 15 por 9,5, colesterol "
                         "acima do normal, glicose normal, fuma, não bebe e é sedentário. Qual o risco?",
    "dados faltando": "Mulher, 61 anos, pressão 14 por 9. Devo encaminhar?",
    "pergunta sobre o modelo": "O modelo diz que fumar reduz o risco? Isso está certo?",
}
if TEM_CHAVE:
    agente = criar_agente()
    for nome, texto in casos.items():
        resposta, chamadas = conversar(agente, [{"role": "user", "content": texto}])
        print(f"=== {nome}\nusuário: {texto}")
        for c in chamadas:
            print(f"  [ferramenta] {c['ferramenta']}")
        print(f"agente: {resposta}\n")
else:
    print("Pulado: defina OPENAI_API_KEY no arquivo .env para executar esta seção.")

=== paciente completo
usuário: Homem de 58 anos, 1,72 m e 94 kg, pressão 15 por 9,5, colesterol acima do normal, glicose normal, fuma, não bebe e é sedentário. Qual o risco?
  [ferramenta] converter_pressao
  [ferramenta] avaliar_paciente
agente: Os dados extraídos foram: 58 anos, masculino, 172 cm, 94 kg, pressão 150/95, colesterol 2, glicose 1, fumante sim, consome álcool não, fisicamente ativo não.

A probabilidade de doença cardiovascular é de 84,9%. A decisão de triagem é encaminhar para investigação, pois está acima do limiar de 38,2%. 

Os fatores que mais pesaram foram:
- Pressão sistólica: 150 mmHg, que aumenta o risco em 22,5 pontos percentuais.
- Colesterol acima do normal, que aumenta o risco em 3,0 pontos percentuais.
- Peso de 94 kg, que aumenta o risco em 2,1 pontos percentuais.

Lembre-se de que a decisão final deve ser tomada pela equipe clínica.



=== dados faltando
usuário: Mulher, 61 anos, pressão 14 por 9. Devo encaminhar?
agente: Faltam alguns dados para que eu possa avaliar a situação. Por favor, informe também a altura (em cm), peso (em kg), colesterol, glicose, se a paciente é fumante, se consome álcool e se é fisicamente ativa.



=== pergunta sobre o modelo
usuário: O modelo diz que fumar reduz o risco? Isso está certo?
  [ferramenta] consultar_achados
agente: O modelo indica que fumar e consumir álcool estão associados a um risco menor, mas isso é um artefato conhecido da base de dados. Esse resultado não reflete um efeito protetor real; fumar e beber continuam sendo fatores de risco cardiovascular estabelecidos. O que ocorre é que esses hábitos autodeclarados podem estar confundidos com outras variáveis, como sexo e idade. Portanto, é importante considerar que fumar e beber não são comportamentos saudáveis.



### Um erro real que os testes revelaram

Na primeira versão, o LLM (`gpt-4o-mini`) convertia a pressão sozinho e, em 2 de 3
tentativas com o mesmo texto, leu "15 por 9,5" como **155/95** em vez de 150/95. A
temperatura 0 não garantiu a mesma resposta. A correção seguiu o princípio do agente:
**o que pode ser calculado com certeza vai para o código**. A ferramenta
`converter_pressao` faz a conversão com uma regra fixa, e o agente passou a começar cada resposta
listando os dados que usou, para que a equipe perceba qualquer erro de extração.

## Conclusões

* O agente **não substitui** a rede: ele é uma camada de linguagem por cima dela. A
  previsão continua vindo do modelo avaliado nos notebooks 02–05, com o mesmo limiar e as
  mesmas métricas.
* A divisão de trabalho reduz riscos típicos de LLMs em saúde: o LLM não inventa números
  (a ferramenta calcula), não assume dados ausentes (as instruções mandam perguntar) e é
  avisado de extrapolações (a ferramenta checa as faixas do treino).
* `consultar_achados` é uma versão mínima de **RAG**: recupera trechos da documentação
  do projeto por palavras-chave. Com mais documentos, seria trocada por embeddings e um
  banco vetorial, como nos slides.
* Na aplicação Streamlit, a aba **Assistente** usa este mesmo agente e mostra cada chamada
  de ferramenta, para que a equipe veja de onde veio o número.